# RoPE Tuning with GGUF Model Surgery

This notebook demonstrates tuning RoPE parameters of a GGUF model for improved context length handling.

In [ ]:
from evolution.gguf.surgeon import ModelSurgeon
from evolution.gguf.rope import RoPEConfig

## 1. Load Model

Initialize the model surgeon and load RoPE parameters:

In [ ]:
# Initialize surgeon
surgeon = ModelSurgeon()

# Get current RoPE config
model_path = "model.gguf"
rope_config = surgeon.get_rope_config(model_path)

print("Current RoPE parameters:")
print(f"Base freq: {rope_config.freq_base}")
print(f"Scale: {rope_config.freq_scale}")
print(f"Dimensions: {rope_config.dimensions}")

## 2. Configure New RoPE Parameters

Define the new RoPE configuration:

In [ ]:
# Define new config
new_config = RoPEConfig(
    freq_base=10000,
    freq_scale=1.0,
    dimensions=128
)

print("New RoPE parameters:")
print(f"Base freq: {new_config.freq_base}")
print(f"Scale: {new_config.freq_scale}")
print(f"Dimensions: {new_config.dimensions}")

## 3. Preview Changes

Generate a preview of the RoPE tuning effects:

In [ ]:
# Preview changes
preview = surgeon.preview(model_path, {
    "rope": new_config.dict()
})

print(f"Will modify {preview.changed_tensors} tensors")
print(f"Size delta: {preview.bytes_delta_mb:.1f}MB")
print(f"Snapshot ID: {preview.snapshot_id}")

## 4. Apply RoPE Tuning

Execute the RoPE parameter update:

In [ ]:
# Apply new RoPE config
out_path = "model.rope.gguf"
result = surgeon.tune_rope(
    model_path,
    new_config,
    out_path
)

print("RoPE tuning complete!")
print(f"Output model: {out_path}")

## 5. Evaluate Effects

Run validation with focus on context handling:

In [ ]:
# Validate
metrics = surgeon.validate(
    out_path,
    "eval/context_length/*",
    focus=["attention", "context"]
)

print("Validation Results:")
print(f"Attention Score: {metrics.attention:.1%}")
print(f"Context Retention: {metrics.context:.1%}")
print(f"Drift: {metrics.drift:.1%}")

if metrics.passes_gates():
    print("\n✅ All validation gates passed")
else:
    print("\n❌ Failed validation gates")

## 6. Embed Provenance

Add provenance data for the RoPE tuning:

In [ ]:
# Sign model
checksum, signature = surgeon.embed_provenance(
    out_path,
    {
        "type": "rope_tune",
        "params": new_config.dict()
    }
)

print(f"Model signed with checksum: {checksum}")
print(f"Signature: {signature}")